# 1. Imports

In [37]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# 2. Connection

In [38]:
load_vars = load_dotenv("../src/.env")
if load_vars:
    print("Variáveis de ambiente carregadas com sucesso")
else:
    print("Arquivo .env não encontrado!")

Variáveis de ambiente carregadas com sucesso


In [39]:
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=tcp:127.0.0.1,1433;"
    "DATABASE=MES_Core;"
    "UID=sa;"
    f"PWD={os.environ['DB_PASSWORD']};"
    "TrustServerCertificate=yes;"
    "Encrypt=no;"
)

In [40]:
connection_url = f"mssql+pyodbc:///?odbc_connect={quote_plus(connection_string)}"

In [41]:
engine = create_engine(connection_url)

# 3. Pegando todas tabelas

In [42]:
tables = [
    'dados_receitas',
    'fila_batch_ids',
    'fila_paradas',
    'ihms',
    'linhas_producao',
    'logs_registradores',
    'notificacoes',
    'ordens_servico',
    'paradas',
    'parametros',
    'receitas',
    'registradores',
    'sistemas'
]

In [43]:
dicionario_dfs = dict()
for table in tables:
    df_apoio = pd.read_sql(f"SELECT * FROM {table}", engine)
    if len(df_apoio) > 0:
        dicionario_dfs[table] = df_apoio
    del df_apoio

In [44]:
# for key in dicionario_dfs.keys():
#     print(key)
#     print(dicionario_dfs[key].head())
#     print('------')
    

# 4. Tratando informação

In [45]:
df_interesse = dicionario_dfs['logs_registradores'].merge(dicionario_dfs['ihms'].rename(columns={'id':'id_ihm'})[['id_ihm', 'nome_maquina']], how='left', on='id_ihm')

In [46]:
df_interesse = df_interesse.merge(dicionario_dfs['registradores'].rename(columns={'id':'id_registrador'})[['id_registrador', 'descricao']], how='left', on='id_registrador')

In [47]:
df_interesse = df_interesse[['nome_maquina', 'descricao', 'datahora', 'valor_bruto']]

In [48]:
df_interesse = df_interesse.pivot_table(index=['nome_maquina','datahora'], columns='descricao', values='valor_bruto', aggfunc='first').reset_index()

In [49]:
df_interesse = df_interesse.sort_values('datahora')
df_interesse.reset_index(drop=True, inplace=True)

In [50]:
df_interesse.columns

Index(['nome_maquina', 'datahora', 'engenheiro', 'manutentor', 'motivo_parada',
       'operador', 'produzido', 'reprovado', 'status_maquina',
       'total_produzido'],
      dtype='object', name='descricao')

In [51]:
depara_status_maquina = {
    '0': 'Parada',
    '1': 'Passar Padrão',
    '49': 'Produzindo',
    '4': 'Limpeza'
}

In [52]:
df_interesse['status_maquina'] = df_interesse['status_maquina'].map(depara_status_maquina)

In [54]:
df_interesse = df_interesse[df_interesse['nome_maquina']=='MAQ1']

In [55]:
relatorio = "Iniciando relatório..."
colunas_interesse = ['engenheiro', 'manutentor', 'motivo_parada',
       'operador', 'produzido', 'reprovado', 'status_maquina',
       'total_produzido']
dicionario_controle = dict()
for i, row in df_interesse.iterrows():
    relatorio += f"\n\n**{row['datahora']}**" 
    if i > 0:
        relatorio += "\nMudanças de Estado"
    for col in colunas_interesse:
        if col not in dicionario_controle.keys():
            relatorio += f"\n{col} (Estado inicial): {row[col]}"
            dicionario_controle[col] = row[col]
        else:
            if dicionario_controle[col] != row[col]:
                relatorio += f"\n{col}: {row[col]}"
                dicionario_controle[col] = row[col]

In [56]:
print(relatorio)

Iniciando relatório...

**2025-11-25 19:58:57.623000**
engenheiro (Estado inicial): 0
manutentor (Estado inicial): 0
motivo_parada (Estado inicial): 49
operador (Estado inicial): 7
produzido (Estado inicial): 59
reprovado (Estado inicial): 13
status_maquina (Estado inicial): Produzindo
total_produzido (Estado inicial): 72


In [57]:
df_interesse#.head()

descricao,nome_maquina,datahora,engenheiro,manutentor,motivo_parada,operador,produzido,reprovado,status_maquina,total_produzido
0,MAQ1,2025-11-25 19:58:57.623,0,0,49,7,59,13,Produzindo,72


In [34]:
df_interesse.shape

(41, 10)

In [35]:
def get_metrics_machine(df_timeline, machine='MAQ1'):
    
    first_register = df_timeline[df_timeline['datahora']==df_timeline['datahora'].min()]
    last_register = df_timeline[df_timeline['datahora']==df_timeline['datahora'].max()]
    status = last_register['status_maquina'].to_list()[0]
    operador = last_register['operador'].to_list()[0]
    manutentor = last_register['manutentor'].to_list()[0]
    produzido = last_register['produzido'].to_list()[0]
    reprovado = last_register['reprovado'].to_list()[0]
    total = last_register['total_produzido'].to_list()[0]
    
    # OEE = Disponibilidade * Performance * Qualidade
    
    # Disponibilidade = Tempo produzido / Tempo programado para produzir
    lista_produzido = []
    status_antigo = ""
    inicio = None
    fim = None
    for i, row in df_timeline.iterrows():
        if status_antigo != 'Produzindo' and row['status_maquina']=='Produzindo':
            inicio = row['datahora']
        elif (status_antigo == 'Produzindo' and row['status_maquina']!='Produzindo') or (status_antigo == 'Produzindo' and row['status_maquina']=='Produzindo' and row['datahora']==last_register['datahora'].to_list()[0]):
            fim = row['datahora']
        if inicio and fim:
            lista_produzido.append((inicio, fim))
            inicio = None
            fim = None            
        status_antigo = row['status_maquina']
    tempo_produzido = sum([y.total_seconds() for y in [x[1] - x[0] for x in lista_produzido]])
    tempo_programado = (last_register['datahora'].to_list()[0] - first_register['datahora'].to_list()[0]).total_seconds()
    disponibilidade = tempo_produzido / tempo_programado
        
    # Performance = Produção Real / Produção Teórica
    meta = (tempo_programado // 1) # Considerando que a cada 1 s é feito uma peça
    performance = int(total) / meta
    
    # Qualidade = Peça Boas / Total de peças
    qualidade = int(produzido) / int(total)
    
    oee = disponibilidade * performance * qualidade
    # OEE, Qualidade, Eficiencia, Meta, Acumulado, Operador, Manutentor, Status
    
    
    return {
        'status': status,
        'oee': round(100 * oee, 2),
        'eficiencia': performance, # eficiencia é isso mesmo?
        'qualidade': round(100 * qualidade, 2),
        'meta': meta,
        'produzido': produzido,
        'reprovado': reprovado,
        'total': total,
        'operador': operador,
        'manutentor': manutentor
    }
    

In [36]:
get_metrics_machine(df_interesse)

{'status': 'Produzindo',
 'oee': 0.18,
 'eficiencia': 0.01842374616171955,
 'qualidade': 9.72,
 'meta': 3908.0,
 'produzido': '7',
 'reprovado': '59',
 'total': '72',
 'operador': '7',
 'manutentor': '0'}